In [ ]:
# ========== 测试两阶段rewrite ==========
# 导入新增的函数
from vector import rewrite_for_retrieval, sparse_search, router, rewrite, hybrid_search

# 测试query
original_query = "合诚工程咨询集团有多少名员工？"


# 阶段1: 补全省略/代词（这里没有历史，所以基本不变）
query_step1 = rewrite(original_query, history=[])
print(f"  原始: {original_query}")
print(f"  补全: {query_step1}")

# 阶段2: 路由确定文档
target_docs = router(query_step1)
print(f"  目标文档: {target_docs}")

# 阶段3: 精简优化
query_step2 = rewrite_for_retrieval(query_step1)
print(f"  优化后: {query_step2}")

# 使用原始query检索
print(f"\n使用原始query: '{original_query}'")
results_original = hybrid_search(original_query, top_k=100, source_filter=target_docs[0] if target_docs != [None] else None)

# 使用优化后query检索
print(f"\n使用优化后query: '{query_step2}'")
results_optimized = hybrid_search(query_step2, top_k=100, source_filter=target_docs[0] if target_docs != [None] else None)

# 查找目标ID在两次检索中的排名
target_id = 967

def find_rank(results, target_id):
    for rank, (doc_id, text, score) in enumerate(results, 1):
        if doc_id == target_id:
            return rank, score
    return None, None

rank_original, score_original = find_rank(results_original, target_id)
rank_optimized, score_optimized = find_rank(results_optimized, target_id)

print(f"\n{'=' * 60}")
print(f"目标ID={target_id}的排名对比:")
print("=" * 60)

# 修复格式化字符串问题
rank_str_orig = f"第 {rank_original} 名" if rank_original else ">100 名"
score_str_orig = f"{score_original:.4f}" if score_original else "N/A"
rank_str_opt = f"第 {rank_optimized} 名" if rank_optimized else ">100 名"
score_str_opt = f"{score_optimized:.4f}" if score_optimized else "N/A"

print(f"原始query:   排名{rank_str_orig} | BM25得分: {score_str_orig}")
print(f"优化后query: 排名{rank_str_opt} | BM25得分: {score_str_opt}")

if rank_original and rank_optimized:
    improvement = rank_original - rank_optimized
    if improvement > 0:
        print(f"\n排名提升了 {improvement} 位！")
    elif improvement < 0:
        print(f"\n 排名下降了 {abs(improvement)} 位")
    else:
        print(f"\n排名无变化")